In [11]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd


import sqlite3

def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    query = f"""
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{{digit}}') as api,pageName as page_ame,
json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].experimentId') experiment_id,
type,uid,date_format(__time__, '%Y%m%d') as ds,count(1) search_times,
array_join(array_agg(distinct json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].variantId')),',') variant_list
from log group by 1,2,3,4,5,6 limit 1000000"""
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    _df["variant_list"] = _df["variant_list"].fillna("none")

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    return _df


all_user_variant_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    df = get_user_variant_of_date_from_sls(current_date)
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_variant_df.head(10)

,api,page_ame,experiment_id,type,uid,ds,search_times,variant_list
0,/product/{digit}/{digit},/search/goods,product_search_rerank,a,298715,20250101,9,V5
1,/product/{digit}/{digit},/search/goods,product_search_rerank,a,351254,20250101,1,V3
2,/product/{digit}/{digit},/search/goods,product_search_rerank,a,465176,20250101,5,V3
3,/product/{digit}/{digit},/search/goods,product_search_rerank,a,391325,20250101,17,V2
4,/product/{digit}/{digit},/search/goods,product_search_rerank,a,182144,20250101,7,V4
5,/product/{digit}/{digit},/search/goods,product_search_rerank,a,491199,20250101,1,V3
6,/product/{digit}/{digit},/search/goods,product_search_rerank,a,104596,20250101,2,V5
7,/product/{digit}/{digit},/search/goods,product_search_rerank,a,463973,20250101,1,V5
8,/product/{digit}/{digit},/search/goods,product_search_rerank,a,400475,20250101,9,V4
9,/product/{digit}/{digit},/search/goods,product_search_rerank,a,213818,20250101,2,V5
